# Notebook 2: Biểu diễn lời giải và hàm mục tiêu

Đi kèm [Phần A, Mục 4.1–4.2](../../docs/A-translation/04-problem-transformation.md): Ví dụ 4, 5, 6 (biến nguyên), Ví dụ 9 (MAX-SAT), Ví dụ 10 (hàm số nguyên), Ví dụ 11 (tô màu đồ thị). Mọi kết quả đều được đối chiếu bằng duyệt vét cạn.

In [1]:
import itertools, sys
from collections import defaultdict
import sympy as sp
sys.path.insert(0, "../../code")
from qco_utils import integer_encoding, multilinear, evaluate, brute_force, penalty_constant

## 1. Biến nguyên → biến nhị phân (Ví dụ 4, 5, 6)

Hàm `integer_encoding(a, c)` trả về $a$ và các trọng số của $z = a + \sum_j w_j x_j$ theo phương trình (20). Với mỗi ví dụ ta kiểm tra rằng tập giá trị $z$ có thể biểu diễn đúng bằng $[a, c]$.

In [2]:
def reachable(a, ws):
    return sorted({a + sum(w * xi for w, xi in zip(ws, x)) for x in itertools.product((0, 1), repeat=len(ws))})

for name, (lo, hi), expected in [
    ("Ví dụ 4: z in [0, 7]",  (0, 7),  [1, 2, 4]),
    ("Ví dụ 5: z in [0, 5]",  (0, 5),  [1, 2, 2]),
    ("Ví dụ 6: z in [-3, 3]", (-3, 3), [1, 2, 3]),
]:
    a, ws = integer_encoding(lo, hi)
    vals = reachable(a, ws)
    print(f"{name:24s} a={a:3d} trọng số={ws}  (bài báo: {expected})  giá trị biểu diễn được: {vals[0]}..{vals[-1]}")
    assert ws == expected and vals == list(range(lo, hi + 1))

Ví dụ 4: z in [0, 7]     a=  0 trọng số=[1, 2, 4]  (bài báo: [1, 2, 4])  giá trị biểu diễn được: 0..7
Ví dụ 5: z in [0, 5]     a=  0 trọng số=[1, 2, 2]  (bài báo: [1, 2, 2])  giá trị biểu diễn được: 0..5
Ví dụ 6: z in [-3, 3]    a= -3 trọng số=[1, 2, 3]  (bài báo: [1, 2, 3])  giá trị biểu diễn được: -3..3


Mỗi giá trị của $z$ có thể có nhiều biểu diễn (ví dụ $z = 3$ trong Ví dụ 5). Điều này đúng như bài báo nêu, và không gây hại vì các biểu diễn khác nhau cho cùng một giá trị mục tiêu.

In [3]:
a, ws = integer_encoding(0, 5)
reps = defaultdict(list)
for x in itertools.product((0, 1), repeat=3):
    reps[a + sum(w * xi for w, xi in zip(ws, x))].append(x)
print(dict(reps))

{0: [(0, 0, 0)], 2: [(0, 0, 1), (0, 1, 0)], 4: [(0, 1, 1)], 1: [(1, 0, 0)], 3: [(1, 0, 1), (1, 1, 0)], 5: [(1, 1, 1)]}


## 2. MAX-SAT: Ví dụ 9, phương trình (25)

Clause $c$ không được thỏa chỉ khi mọi literal sai. Với literal dương $x_j$ ta dùng $(1 - x_j)$, với literal âm $\neg x_j$ ta dùng $x_j$.

In [4]:
x1, x2, x3 = sp.symbols("x1 x2 x3")
clauses = [
    ((x1, True), (x2, False), (x3, True)),    # x1 or not x2 or x3
    ((x1, True), (x2, True),  (x3, False)),   # x1 or x2 or not x3
    ((x1, False), (x2, False), (x3, True)),   # not x1 or not x2 or x3
]
def unsat_poly(clause):
    expr = 1
    for var, positive in clause:
        expr *= (1 - var) if positive else var
    return expr

f_expr = sum(unsat_poly(c) for c in clauses)
f_poly = multilinear(f_expr, [x1, x2, x3])
print("f(x) =", sp.expand(f_expr))
print("khớp phương trình (25):", sp.expand(f_expr - (x1*x2*x3 - 2*x2*x3 - x1*x3 + x2 + x3)) == 0)

# đối chiếu với việc đếm trực tiếp số clause không được thỏa
def count_unsat(x):
    xs = dict(zip((x1, x2, x3), x))
    return sum(0 if any((xs[v] == 1) == pos for v, pos in c) else 1 for c in clauses)
assert all(evaluate(f_poly, x) == count_unsat(x) for x in itertools.product((0, 1), repeat=3))
best, arg = brute_force(f_poly, 3)
print("giá trị nhỏ nhất:", best, "| các phép gán thỏa mọi clause:", arg)

f(x) = x1*x2*x3 - x1*x3 - 2*x2*x3 + x2 + x3
khớp phương trình (25): True
giá trị nhỏ nhất: 0 | các phép gán thỏa mọi clause: [(0, 0, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 1)]


Có **5** phép gán thỏa mọi clause: `000`, `011`, `100`, `101`, `111`. Con số này quan trọng khi đọc kết quả của Hình 4 và Bảng 2 ở [Notebook 4](04-quantum-annealer-maxsat.ipynb).

## 3. Hàm số nguyên: Ví dụ 10

$f(z) = z^3 - 6z$ với $z \in [-3, 3]$ và $z = -3 + x_1 + 2x_2 + 3x_3$. Ta khai triển và áp dụng $x^k = x$.

In [5]:
x1, x2, x3 = sp.symbols("x1 x2 x3")
z = -3 + x1 + 2*x2 + 3*x3
f_expr = z**3 - 6*z
f_poly = multilinear(f_expr, [x1, x2, x3])
names = {0: "x1", 1: "x2", 2: "x3"}
for S, c in sorted(f_poly.items(), key=lambda kv: (len(kv[0]), sorted(kv[0]))):
    print(f"{str(c):>5}  *","*".join(names[i] for i in sorted(S)) or "1")

# so với đáp án trong bài báo: -9 + 13x1 + 14x2 + 9x3 - 18x1x2 - 18x1x3 - 18x2x3 + 36x1x2x3
paper = {frozenset(): -9, frozenset({0}): 13, frozenset({1}): 14, frozenset({2}): 9,
         frozenset({0, 1}): -18, frozenset({0, 2}): -18, frozenset({1, 2}): -18, frozenset({0, 1, 2}): 36}
print("khớp bài báo:", f_poly == paper)
# và khớp với hàm gốc z^3 - 6z trên mọi tổ hợp
fz = lambda t: t**3 - 6*t
assert all(evaluate(f_poly, x) == fz(-3 + x[0] + 2*x[1] + 3*x[2]) for x in itertools.product((0, 1), repeat=3))
print("giá trị nhỏ nhất:", brute_force(f_poly, 3))

   -9  * 1
   13  * x1
   14  * x2
    9  * x3
  -18  * x1*x2
  -18  * x1*x3
  -18  * x2*x3
   36  * x1*x2*x3
khớp bài báo: True
giá trị nhỏ nhất: (-9, [(0, 0, 0)])


## 4. Tô màu đồ thị: Ví dụ 7 và 11

Đồ thị đường đi gồm 3 nút, các cạnh $(0,1)$ và $(1,2)$, với 3 màu. Hàm mục tiêu (28) là $\sum_{(j,k)\in E}\sum_{\ell} x_{j,\ell} x_{k,\ell}$, cộng hàm phạt one-hot $P\sum_j (\sum_\ell x_{j,\ell} - 1)^2$ cho ràng buộc (22).

In [6]:
nodes, edges, colors = 3, [(0, 1), (1, 2)], 3
xs = sp.symbols(f"y0:{nodes*colors}")
X = lambda j, l: xs[j*colors + l]

objective = sum(X(j, l) * X(k, l) for (j, k) in edges for l in range(colors))
onehot = sum((sum(X(j, l) for l in range(colors)) - 1)**2 for j in range(nodes))
obj_poly = multilinear(objective, xs)
P = penalty_constant(obj_poly)                     # theo (36), áp dụng cho phần mục tiêu
total = multilinear(objective + P * onehot, xs)
print("P =", P)

best, arg = brute_force(total, nodes*colors)
def decode(x):
    return [x[j*colors:(j+1)*colors].index(1) for j in range(nodes)]
print("giá trị nhỏ nhất:", best, "| số nghiệm tối ưu:", len(arg))
assert all(sum(x[j*colors:(j+1)*colors]) == 1 for x in arg for j in range(nodes))        # mọi nghiệm tối ưu đều khả thi
assert all(decode(x)[0] != decode(x)[1] and decode(x)[1] != decode(x)[2] for x in arg)
print("ví dụ vài cách tô:", [decode(x) for x in arg[:4]])

P = 7
giá trị nhỏ nhất: 0 | số nghiệm tối ưu: 12
ví dụ vài cách tô: [[2, 1, 2], [2, 1, 0], [2, 0, 2], [2, 0, 1]]


Có $3 \cdot 2 \cdot 2 = 12$ cách tô hợp lệ cho đường đi 3 nút với 3 màu, và duyệt vét cạn trên 9 biến nhị phân cho đúng 12 nghiệm tối ưu, tất cả đều thỏa ràng buộc one-hot và không có hai nút kề nhau cùng màu.